In [1]:
pip install trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 540.5/540.5 kB 8.6 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install -q qwen-vl-utils

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 MB 50.9 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install -U bitsandbytes>=0.46.

Note: you may need to restart the kernel to use updated packages.


# imports

In [4]:
from transformers import BitsAndBytesConfig
from transformers import AutoProcessor, AutoModelForImageTextToText, TrainingArguments
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer
from transformers import Trainer

In [5]:
from kaggle_secrets import UserSecretsClient
import wandb

user_secrets = UserSecretsClient()
wandb_key = user_secrets.get_secret("W&B")

wandb.login(key=wandb_key)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: mishraarnav32 (mishraarnav32-manipal) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

# W&B 

In [6]:
wandb.init(
    project="openpack-vlm-finetune",
    name="qwen2vl-qlora-u0209",
    config={
        "model": "Qwen2-VL-2B-Instruct",
        "method": "QLoRA",
        "dataset": "OpenPack-U0209",
        "lora_r": 16,
        "lora_alpha": 32,
        "batch_size": 1,
        "grad_accum": 16,
        "epochs": 10,
        "lr": 2e-4,
    }
)
report_to="wandb",

wandb: setting up run 7urb878n
wandb: Tracking run with wandb version 0.24.0
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260224_063342-7urb878n
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run qwen2vl-qlora-u0209
wandb: ⭐️ View project at https://wandb.ai/mishraarnav32-manipal/openpack-vlm-finetune
wandb: 🚀 View run at https://wandb.ai/mishraarnav32-manipal/openpack-vlm-finetune/runs/7urb878n


# Calculations for sanity test

In [7]:
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name} — {props.total_memory / 1e9:.1f} GB")

CUDA available: True
GPU count: 2
  GPU 0: Tesla T4 — 15.6 GB
  GPU 1: Tesla T4 — 15.6 GB


In [8]:
model_base_4bit  = 2.0   # GB — Qwen2.5-VL-2B at 4-bit
lora_adapters    = 0.3   # GB — LoRA rank=16, alpha=32
frames_per_clip  = 8     # Sampled frames per 5-sec clip
frame_tokens     = 256   # Visual tokens per frame
batch_size       = 2
token_hidden_dim = 1536  # Qwen2.5-VL-2B hidden size

activation_gb = (frames_per_clip * frame_tokens * batch_size * token_hidden_dim * 2) / 1e9
total_vram_gb = model_base_4bit + lora_adapters + (activation_gb * 0.4)

print(f"Model (4-bit):        {model_base_4bit:.2f} GB")
print(f"LoRA adapters:        {lora_adapters:.2f} GB")
print(f"Activations (raw):    {activation_gb:.2f} GB")
print(f"Activations (w/ GC):  {activation_gb * 0.4:.2f} GB")
print(f"─────────────────────────────")
print(f"Estimated VRAM:       {total_vram_gb:.2f} GB")
print(f"T4 limit:             16.00 GB")
print(f"Status: {'✅ FITS' if total_vram_gb < 16 else '❌ OOM RISK'}")

Model (4-bit):        2.00 GB
LoRA adapters:        0.30 GB
Activations (raw):    0.01 GB
Activations (w/ GC):  0.01 GB
─────────────────────────────
Estimated VRAM:       2.31 GB
T4 limit:             16.00 GB
Status: ✅ FITS


In [9]:
model_base_fp16  = 4.0   # GB — Qwen2.5-VL-2B at fp16 (no quantization)
lora_adapters    = 0.3   # GB — LoRA rank=16, alpha=32
frames_per_clip  = 8
frame_tokens     = 256
batch_size       = 2
token_hidden_dim = 1536

activation_gb = (frames_per_clip * frame_tokens * batch_size * token_hidden_dim * 2) / 1e9
total_vram_gb = model_base_fp16 + lora_adapters + (activation_gb * 0.4)

print(f"Model (fp16):         {model_base_fp16:.2f} GB")
print(f"LoRA adapters:        {lora_adapters:.2f} GB")
print(f"Activations (raw):    {activation_gb:.2f} GB")
print(f"Activations (w/ GC):  {activation_gb * 0.4:.2f} GB")
print(f"─────────────────────────────")
print(f"Estimated VRAM:       {total_vram_gb:.2f} GB")
print(f"T4 limit:             16.00 GB")
print(f"Status: {'✅ FITS' if total_vram_gb < 16 else '❌ OOM RISK'}")


Model (fp16):         4.00 GB
LoRA adapters:        0.30 GB
Activations (raw):    0.01 GB
Activations (w/ GC):  0.01 GB
─────────────────────────────
Estimated VRAM:       4.31 GB
T4 limit:             16.00 GB
Status: ✅ FITS


## Technically we can even go with LoRA, but since the assignment specifically asked for 4bit quantised , so we go with QLoRA

# Using BitsAndBytes for QLORA

In [10]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

In [11]:
model = AutoModelForImageTextToText.from_pretrained(
    "Qwen/Qwen2-VL-2B-Instruct",
    torch_dtype=torch.float16,
    device_map="auto"
)

config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/272 [00:00<?, ?B/s]

In [12]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)


In [13]:
print(f"VRAM after QLoRA: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

VRAM after QLoRA: 2.17 GB


In [14]:
training_args = TrainingArguments(
    output_dir="./checkpoints",
    num_train_epochs=10, # we are keeping this low because we only have 8 sample dataset
    per_device_train_batch_size=1,      # reduced from 2
    gradient_accumulation_steps=16,     # keep effective batch = 16
    gradient_checkpointing=True,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    bf16=True,
    logging_steps=1,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=3,
    report_to="none",
    dataloader_num_workers=0,
    remove_unused_columns=False,
    resume_from_checkpoint=True,
    ddp_find_unused_parameters=False,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [15]:
# After training, load your QLoRA finetuned model instead of base
from peft import PeftModel

base_model = AutoModelForImageTextToText.from_pretrained(
    "Qwen/Qwen2-VL-2B-Instruct",
    torch_dtype=torch.float16,
    device_map="auto"
)

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

# preparing our dataset 

In [16]:
import os

def print_tree(root_dir):
    for root, dirs, files in os.walk(root_dir):
        level = root.replace(root_dir, "").count(os.sep)
        indent = "│   " * level
        print(f"{indent}📁 {os.path.basename(root)}/")

        sub_indent = "│   " * (level + 1)
        for f in files:
            print(f"{sub_indent}📄 {f}")

# usage
print_tree(r"/kaggle/input/datasets/arnavmishra6996/vlm-qlora-dataset")


📁 vlm-qlora-dataset/
│   📁 S0500/
│   │   📄 frame_000006.jpg
│   │   📄 frame_000010.jpg
│   │   📄 frame_000008.jpg
│   │   📄 frame_000004.jpg
│   │   📄 frame_000012.jpg
│   │   📄 frame_000001.jpg
│   │   📄 frame_000002.jpg
│   │   📄 frame_000013.jpg
│   │   📄 frame_000000.jpg
│   │   📄 frame_000011.jpg
│   │   📄 frame_000007.jpg
│   │   📄 frame_000014.jpg
│   │   📄 frame_000009.jpg
│   │   📄 frame_000003.jpg
│   │   📄 frame_000005.jpg
│   📁 training_data_samples/
│   │   📄 sample_017.json
│   │   📄 sample_005.json
│   │   📄 sample_019.json
│   │   📄 sample_003.json
│   │   📄 sample_011.json
│   │   📄 sample_015.json
│   │   📄 sample_007.json
│   │   📄 sample_018.json
│   │   📄 sample_013.json
│   │   📄 sample_000.json
│   │   📄 sample_010.json
│   │   📄 sample_012.json
│   │   📄 sample_016.json
│   │   📄 sample_008.json
│   │   📄 sample_009.json
│   │   📄 sample_006.json
│   │   📄 sample_002.json
│   │   📄 sample_001.json
│   │   📄 sample_004.json
│   │   📄 sample_014.json


In [17]:
from transformers import AutoProcessor
processor = AutoProcessor.from_pretrained("Qwen/Qwen2-VL-2B-Instruct")
print("Processor loaded.")

preprocessor_config.json:   0%|          | 0.00/347 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Processor loaded.


In [18]:
import json
from pathlib import Path

DATASET_PATH = Path("/kaggle/input/datasets/arnavmishra6996/vlm-qlora-dataset/training_data_samples")
FRAMES_PATH  = Path("/kaggle/input/datasets/arnavmishra6996/vlm-qlora-dataset/S0500")

all_pairs = []
for json_file in sorted(DATASET_PATH.glob("sample_*.json")):
    with open(json_file) as f:
        pair = json.load(f)
    # Fix Windows backslashes and grab filename only
    pair["sampled_frame_paths"] = [
        str(FRAMES_PATH / Path(fp.replace("\\", "/")).name)
        for fp in pair["sampled_frame_paths"]
    ]
    all_pairs.append(pair)

print(f"Loaded {len(all_pairs)} pairs")
for fp in all_pairs[0]["sampled_frame_paths"]:
    print(f"  {'✅' if Path(fp).exists() else '❌'} {fp}")

Loaded 20 pairs
  ✅ /kaggle/input/datasets/arnavmishra6996/vlm-qlora-dataset/S0500/frame_000000.jpg
  ✅ /kaggle/input/datasets/arnavmishra6996/vlm-qlora-dataset/S0500/frame_000001.jpg
  ✅ /kaggle/input/datasets/arnavmishra6996/vlm-qlora-dataset/S0500/frame_000002.jpg
  ✅ /kaggle/input/datasets/arnavmishra6996/vlm-qlora-dataset/S0500/frame_000003.jpg
  ✅ /kaggle/input/datasets/arnavmishra6996/vlm-qlora-dataset/S0500/frame_000004.jpg
  ✅ /kaggle/input/datasets/arnavmishra6996/vlm-qlora-dataset/S0500/frame_000005.jpg
  ✅ /kaggle/input/datasets/arnavmishra6996/vlm-qlora-dataset/S0500/frame_000006.jpg
  ✅ /kaggle/input/datasets/arnavmishra6996/vlm-qlora-dataset/S0500/frame_000013.jpg


In [19]:
from torch.utils.data import Dataset
from qwen_vl_utils import process_vision_info

class OpenPackDataset(Dataset):
    def __init__(self, pairs, processor):
        self.pairs = pairs
        self.processor = processor

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        pair = self.pairs[idx]
        frame_paths = pair["sampled_frame_paths"]

        messages = [
            {"role": "system", "content": pair["system_prompt"]},
            {
                "role": "user",
                "content": [
                    *[{"type": "image", "image": f"file://{Path(fp).resolve()}"}
                      for fp in frame_paths],
                    {"type": "text", "text": f"Clip ID: {pair['clip_id']}\nAnalyze these frames."},
                ],
            },
            {"role": "assistant", "content": pair["target_json"]},
        ]

        text = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
        image_inputs, video_inputs = process_vision_info(messages)
        inputs = processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
        )
        inputs = {k: v.squeeze(0) for k, v in inputs.items()}
        inputs["labels"] = inputs["input_ids"].clone()
        return inputs

train_dataset = OpenPackDataset(all_pairs, processor)
print(f"Training samples: {len(train_dataset)}")

Training samples: 20


In [20]:
def data_collator(features):
    from torch.nn.utils.rnn import pad_sequence
    import torch

    batch = {}
    for key in features[0].keys():
        tensors = [f[key] for f in features]
        
        if key == "image_grid_thw":
            # Stack grid tensors — must be 2D (N, 3)
            batch[key] = torch.cat([t.reshape(-1, 3) for t in tensors], dim=0)
        elif tensors[0].dim() == 1:
            batch[key] = pad_sequence(tensors, batch_first=True, padding_value=0)
        elif tensors[0].dim() == 2:
            batch[key] = pad_sequence(tensors, batch_first=True, padding_value=0)
        else:
            try:
                batch[key] = torch.stack(tensors)
            except Exception:
                batch[key] = torch.cat(tensors, dim=0)
    
    return batch

# back to QLoRA

In [21]:
model = AutoModelForImageTextToText.from_pretrained(
    "Qwen/Qwen2-VL-2B-Instruct",
    quantization_config=bnb_config,
    device_map="auto"
)

model.gradient_checkpointing_enable()
model.enable_input_require_grads()

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

trainable params: 18,464,768 || all params: 2,227,450,368 || trainable%: 0.8290


In [22]:
train_dataset = OpenPackDataset(all_pairs, processor)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
)

In [23]:
trainer.train()

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


Step,Training Loss
1,17.305857
2,17.307400
3,16.791826
4,13.518875
5,10.714156
6,9.403111
7,8.720733
8,8.383272
9,8.138114
10,7.966228


TrainOutput(global_step=20, training_loss=9.728850412368775, metrics={'train_runtime': 3585.15, 'train_samples_per_second': 0.056, 'train_steps_per_second': 0.006, 'total_flos': 3262947539927040.0, 'train_loss': 9.728850412368775, 'epoch': 10.0})

In [24]:
# Save the final LoRA adapter weights
trainer.save_model("./checkpoints/final")
# Also save the processor
processor.save_pretrained("./checkpoints/final")
print("Model and processor saved to ./checkpoints/final")

Model and processor saved to ./checkpoints/final
